# Dense-16 HELM assumption audit

This notebook deliberately pauses new routing experiments and audits the **vanilla 16-head dense baseline**.

It tests assumptions we have been treating as facts:

- Is the dense model's head-level effective rank actually high?
- Is low raw rank caused by duplicate directions or simply unequal head strength?
- Does collapse exist **before** the output projection, or does `W_O` create/concentrate it?
- Are all 16 heads actually useful to MLM CE?
- Does residual RMS/energy predict exact head importance?
- How quickly does a model trained dense degrade when heads are removed post-hoc?
- Is the 6.5k checkpoint representative, or does rank evolve significantly over training?

The first analysis is a deep audit of one checkpoint. The second tracks rank across several checkpoints using the same validation examples.


In [1]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu]==2.8.0",
                "numpy", "pyarrow==16.1.0", "fsspec", # <-- Pinned pyarrow here
                "protobuf>=5.28.0",
                "datasets>=2.20.0", "transformers", "huggingface_hub>=0.28.0", "wandb", # <-- Added >=2.20.0 to datasets
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

🔍 Starting High-Speed TPU Repair...
🧹 Wiping libraries...
📥 Installing Synced TPU Stack...



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



✅ TPU REPAIR COMPLETE.
⚠️ Click 'Run' -> 'Restart Session' NOW.


## 1. Write the exact vanilla model

This is the dense model source you supplied. The active `HELMBlock` contains no router and all attention heads contribute on every forward pass.


In [2]:
%%writefile model.py

##################################################
# Defines the HELM V1 architecture
# Inherited the PretrainedConfig and PreTrainedModel
# Utilizes many of the concepts found in Nvidia's 2024 nGPT architecture
# Vanilla - Removed Router
# model_repo_id: str = "JamesResearch1216/phase06v4-Vanilla"
# wandb_entity: str = "jhui16-university-of-maryland"
# wandb_project: str = "HELM-v1-10B-Run"
# wandb_name: str = "phase06v4"
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Cast the input to the correct input layer dtype
def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x,w,b)


# Hugging Face Config Class (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm"

    def __init__(

        self,

        # General Model Hyperparameters
        hidden_size = 1024,
        sqrt_hidden_size = 32,
        max_position_embeddings = 4096,
        initializer_range = 0.03125,
        num_hidden_layers = 12,
        num_attention_heads = 16,
        d_head = None,
        rope_theta = 160000,
        intermediate_size = 2816,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        base_lr = 3e-4,
        min_lr = 3e-5,
        weight_decay = 0.0,
        bias = False,
        use_ckpt = False,

        # Tokenization and Data Collator Hyperparameters
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,

        # Router Hyperparameters
        num_router_latents = 4,
        num_permanent_heads = 2,
        selection_threshold = 0.5,
        router_init_scale = 1.0,
        use_sigmoid_scaling = False,
        jitter_noise = 0.01,
        router_grad_clip = 0.05,
        dense_warmup_steps = 0.03,


        # Router Sparsity Hyperparameters
        sparsity_lambda = 0.01,
        sparsity_warm_up_steps = 0.05,
        head_target_min = 4,
        head_target_center = 8,
        head_target_max = 16,
        easiness_cdf_breakpoints = None,
        sparsity_slack_lo = 1.0,
        sparsity_slack_hi = 2.0,

        # Router Auxiliary Hyperparameters:
        aux_coeff_start = 0.02,
        aux_coeff_floor = 0.002,
        aux_anneal_start = 0.08,
        aux_anneal_steps = 0.25,

        # ngpt self attention and ffn hyperparameters
        ngpt_sqk_init_value = 1.0,
        ngpt_sqk_init_scale = 0.03125,
        use_exclusive_attention = True,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.03125,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.03125,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.03125,

        # Passing total step count for warm up step calculations:
        dataset_total_steps = 65000,

        **kwargs
    ):
        # General Model Hyperparameters
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        # Tokenization and Data Collator Hyperparameters
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        # Router Hyperparameters
        self.num_router_latents = num_router_latents
        self.num_permanent_heads = num_permanent_heads
        self.selection_threshold = selection_threshold
        self.router_init_scale = router_init_scale
        self.use_sigmoid_scaling = use_sigmoid_scaling
        self.jitter_noise = jitter_noise
        self.router_grad_clip = router_grad_clip
        self.dense_warmup_steps = int(dense_warmup_steps * dataset_total_steps)

        # Router Sparsity Hyperparameters
        self.sparsity_lambda = sparsity_lambda
        self.sparsity_warm_up_steps = int(sparsity_warm_up_steps * dataset_total_steps)
        self.head_target_min = head_target_min
        self.head_target_center = head_target_center
        self.head_target_max = head_target_max
        self.easiness_cdf_breakpoints = easiness_cdf_breakpoints
        self.sparsity_slack_lo = sparsity_slack_lo
        self.sparsity_slack_hi = sparsity_slack_hi

        # Router Auxiliary Hyperparameters:
        self.aux_coeff_start = aux_coeff_start
        self.aux_coeff_floor = aux_coeff_floor
        self.aux_anneal_start = int(aux_anneal_start * dataset_total_steps)
        self.aux_anneal_steps = int(aux_anneal_steps * dataset_total_steps)

        # ngpt self attention and ffn hyperparameters
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale

        super().__init__(**kwargs)



# Define Embedding Layer
class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id
        )

    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):

        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings




# NOVEL: Multi-Latent Summary Router to decide which heads to use
class HELMMultiViewRouter(nn.Module):

    # Initialize the following:
    #   - Summary Query Matrix (q_down_proj)
    #   - Latent Importance Weights (l_i_weights)
    #   - Router_Init_Scale (tau)
    #   - Linear Router Gate (q_down_proj)
    def __init__(self, config):
        super().__init__()

        # Yoink some things from config
        self.config = config
        self.scale = config.sqrt_hidden_size
        self.num_elastic_candidates = config.num_attention_heads - config.num_permanent_heads


        # Summary Query Matrix size() : [hidden_size, num_router_latents]
        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias = config.bias
        )

        # Latent Importance Weights
        # size() [num_router_latents]
        self.l_i_weights = nn.Parameter(
            torch.ones(config.num_router_latents)
        )

        # Router_Init_Scale size() : [1]
        self.tau = nn.Parameter(torch.tensor(config.router_init_scale))

        # Linear Router Gate size() : [hidden_size, num_attention_heads - num_permanent_heads]
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            config.num_attention_heads - config.num_permanent_heads,
            bias = config.bias
        )

    # Map BT_easiness -> Target # of heads
    # Problem: most of the BT_easiness_score are around .34 mark
    # We need to now center the BT easiness around the median, where a score with 0.34 should be assigned to 8/16 heads should, not 0.5
    def _easiness_to_target(self, sigmoid_scores, easiness_score):

        batch = sigmoid_scores.size(0)
        device = sigmoid_scores.device
        total_heads = self.config.num_attention_heads
        permanent_heads = self.config.num_permanent_heads

        # Get Bucket Target Values (or default to these )
        h_min = float(getattr(self.config, "head_target_min", permanent_heads + 2))
        h_ctr = float(getattr(self.config, "head_target_center", 0.5 * (h_min + total_heads)))
        h_max = float(getattr(self.config, "head_target_max", total_heads))

        # I clamped theses values before, but we'll do it here just in case
        # Size: [batch] (of easiness_score)
        easiness_score = easiness_score.to(torch.float32).view(batch).clamp(0.0,1)
        bp = getattr(self.config, "easiness_cdf_breakpoints", None)

        # Convert the breakpoints to device
        breaks = torch.as_tensor(bp, device = device, dtype=torch.float32)
        num_breaks = breaks.numel() - 1
        pos = torch.searchsorted(breaks, easiness_score, right=True).clamp(1, num_breaks)
        lo = breaks[pos - 1]; hi = breaks[pos]
        frac_in = (easiness_score - lo) / (hi - lo + 1e-8)
        q = ((pos - 1).to(torch.float32) + frac_in) / num_breaks
        q = q.clamp(0.0, 1.0)
        hard = q < 0.5
        t_hard = h_ctr + (h_max - h_ctr) * (0.5 - q) / 0.5
        t_easy = h_ctr + (h_min - h_ctr) * (q - 0.5) / 0.5
        t_total = torch.where(hard, t_hard, t_easy)

        return (t_total - permanent_heads).clamp(0.0, float(total_heads - permanent_heads))

    # Pass in only Hidden States
    # Don't pass in attention mask bc theres no attention here (duh)
    def forward(self, hidden_states, step_tensor, easiness_score):

        #################### FINALIZED LOGIC ####################

        # Write vars for cleaner code
        q_down_proj = self.q_down_proj
        l_i_weights = self.l_i_weights
        tau = self.tau
        q_up_proj = self.q_up_proj
        scale = self.scale
        self.selection_threshold = self.config.selection_threshold

        # Norm Query Matrix
        # Requires .weight since the matrix was defined before
        q_down_proj = justnorm(q_down_proj.weight, dim = 1).to(hidden_states.dtype)

        # Multiply the hidden_state by Down projection (q_down_proj)
        # Call it "scanner"
        # Size: [b, s, hidden_size] * [hidden_size * num_router_latents] = [b, s, num_router_latents]
        scanner = F.linear(hidden_states, q_down_proj)

        # Apply Softmax to entire sequences (sequence level routing)
        scanner_softmax = F.softmax(scale * scanner, dim = 1)

        # Apply Transpose to allow for dimension matching
        # [b, s, num_router_latents] -> [b, num_router_latents, s]
        scanner_softmax = scanner_softmax.transpose(1,2)

        # Create Latent Vectors (Summary of the sequence in 4 vectors)
        # Size: [b,num_router_latents,s] * [b, s, hidden_size] = [b, num_router_latents, hidden_size]
        # Use bmm (batch matrix matric product) b/c [b, n_r_l, s] * [b, s, h_s] (dims don't match up normally)
        # Could've transposed, but this is more memory efficient
        latents = torch.bmm(scanner_softmax, hidden_states)

        # Scale latents by Learnable important parameters (l_i_weights)
        # Softmax them first
        l_i_weights = F.softmax(l_i_weights, dim = 0)

        # Apply l_i_weights to latents
        # Sum the Latents together
        # Size: ([b, num_router_latents, hidden_size] * broadcast [1, num_router_latents, 1]) and sum the latents = [b, 1 (size of pooled_latents when we added them together), hidden_size]
        pooled_latents = (latents * l_i_weights.view(1, -1, 1)).sum(dim=1, keepdim = True)

        # Normalize q_up_proj
        # Requires .weight since the matrix was defined before
        # size [total_elastic_heads, hidden_size]
        q_up_proj = justnorm(q_up_proj.weight, dim = 1).to(pooled_latents.dtype)

        # Multiply the latents by the classifer (q_up_proj)
        # Call it "class_scores"
        # Size: [b, 1, hidden_size] * [hidden_size, total_elastic_heads] = [b, 1, total_elastic_heads]
        class_scores = F.linear(pooled_latents, q_up_proj)

        # Multiply this by Tau (router_init_scale) and ngpt scaler sqrt(hidden_size), or should we???
        class_scores = class_scores * tau

        # Sigmoid Scores
        # Size: still [b, 1, total_elastic_heads], but with sigmoid scores
        sigmoid_scores = torch.sigmoid(class_scores)

        #################### FINALIZED LOGIC ENDS HERE ####################

        # Hard Mask: of 1s and 0s based on whether the sigmoid score > threshold (0.5)
        # Size: [b, 1, total_elastic_heads]
        flat_mask = (sigmoid_scores > self.selection_threshold).float()

        # Dense warmup: ensure all heads are active before dense_warmup_steps
        dense_warmup_steps = self.config.dense_warmup_steps
        in_dense_warmup = step_tensor < dense_warmup_steps
        # Use where pattern to un-mask
        # torch.ones_like (copies all metadata (device, datatype)) ; torch.ones requires you to define all metadata + shape
        flat_mask = torch.where(in_dense_warmup, torch.ones_like(flat_mask), flat_mask)

        # Telemetry hooks
        self.save_flat_mask = flat_mask.detach()
        self.save_sigmoid_scores = sigmoid_scores.detach()

        # use_sigmoid_scaling = True: router_mask = Sigmoid values and 0s (Accuracy)
        # use_sigmoid_scaling = False: router_mask = 1s and 0s (Efficiency)
        flat_mask = flat_mask * sigmoid_scores if self.config.use_sigmoid_scaling else flat_mask

        # Apply STE for Dead Router Heads during backprop
        # Must happen after sigmoid_scaling or else torch could believe the sigmoid scaling are dynamically linked
        # .detach() Ignored during Backprop (Autograd doesn't see anything with.detach(), so when backprop happens, they disappear)
        # Forward pass: flat_mask- sigmoid_scores + sigmoid_scores = flat_mask
        # Backward pass: sigmoid_scores
        flat_mask = flat_mask.detach() - sigmoid_scores.detach() + sigmoid_scores

        # Add attention dimensions: [b, 1, total_elastic_heads] -> [b, num_elastic_heads, 1, 1]
        router_mask = flat_mask.view(flat_mask.size(0), -1, 1, 1)

        # If permanent_heads are used, add the columns
        # size(): [b, num_attention_heads, 1 , 1]
        if (self.config.num_permanent_heads > 0):
            permanent_head_scores = torch.ones(
                flat_mask.size(0),
                self.config.num_permanent_heads,
                1,
                1,
                device=router_mask.device,
                dtype=router_mask.dtype
            )
            router_mask = torch.cat((permanent_head_scores, router_mask), dim = 1)

        #################### LOSS CALCULATIONS ####################
        # In both previous implementations, the model could cheat by turning off all the heads to reduce loss
        # The old formulas were okay for scores passed into variant activations (softmax), but breaks at invariant activation (sigmoid)
        # We need to solve this by redefining how these losses are being calculated

        if self.training:

            # Generate the target-head count from easiness (soft PRIOR; CE can override a wrong label)
            elastic_target_num_heads =  self._easiness_to_target(sigmoid_scores, easiness_score)

            # ########## SPARSITY ##########: Ensure the correct # of heads are being activated

            # Why not count the number of heads to use > 0.5 ? Ans: > produces gradients of 0. Plus, we don't account for on edge heads (i.e .49)
            # By using the sum of the sigmoid scores along the head dimension, the gradient sees the proportion to how "almost on" heads are
            # Just think about this as how many heads should be on?
            num_head_preds = sigmoid_scores.squeeze(1).sum(-1)

            # Define our lower and upper clearances
            slack_lo = self.config.sparsity_slack_lo
            slack_hi = self.config.sparsity_slack_hi

            # WOW check this out:
            # If the computed value is < 0, then it becomes 0 (relu)
            # preds - upperbound for the over (If its less than the upperbound -> 0)
            # lowerbound - preds for the under (If its greater than the lowerbound -> 0)
            # We can define an over or under this way
            over = torch.relu(num_head_preds - (elastic_target_num_heads + slack_hi))
            under = torch.relu((elastic_target_num_heads - slack_lo) - num_head_preds)
            # Squared Sum penalty to extremely punish big changes
            raw_sparsity = (over**2 + under**2).mean()
            # Sparsity warmup
            # Denom: fixed. sparsity starts at 0 and reach 1 when step tensor reaches it
            denom = max(1, self.config.sparsity_warm_up_steps)
            # Ramp: go from 0 -> 1 starting at dense_warm_up -> sparsity_warm_up_steps + dense_warm_up
            ramp = torch.clamp((step_tensor.float() - dense_warmup_steps) / denom, 0.0, 1.0)
            self.sparsity_loss = ramp * self.config.sparsity_lambda * raw_sparsity

            # ########## SPARSITY ENDS ##########

            # ########## AUXILIARY BEGINS ##########
            # aux_loss: ensure that the router doesn't route the same head everytime (even routing)
            # This needs to be designed so its scale invariant (or else collapse will be encouraged)
            # CV^2 (coeff of variation^2) of per-head usage

            # First calculate the average sigmoid value per head in all the seqs in the batch
            # Size: [num_elastic_heads]
            head_avg_scores = sigmoid_scores.squeeze(1).mean(0)
            cv2 = head_avg_scores.var(unbiased = False) / (head_avg_scores.mean()**2 + 1e-6)

            # Yoink from config
            aux_start = self.config.aux_coeff_start
            aux_floor = self.config.aux_coeff_floor
            aux_begin = self.config.aux_anneal_start
            aux_steps = self.config.aux_anneal_steps

            # aux_frac: go from 1 -> 0 starting at aux_anneal_start -> aux_anneal_start + aux_anneal_steps
            aux_frac = torch.clamp((step_tensor.float() - aux_begin) / aux_steps, 0.0, 1.0)
            aux_coeff = aux_start + (aux_floor - aux_start) * aux_frac
            self.aux_loss = aux_coeff * cv2

            # ########## AUXILIARY ENDS ##########

        else:
            self.aux_loss = torch.tensor(0.0, device=hidden_states.device)
            self.sparsity_loss = torch.tensor(0.0, device=hidden_states.device)

        # Cast the router to matching data_type before returning:
        router_mask = router_mask.to(hidden_states.dtype)

        # Return Mask
        # [b, num_attention_heads, 1 , 1]
        return router_mask



# RoPE Class
class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Speicfics about training (masked training)
# MODIFICATION: USE FLEX ATTENTION TO ALLOW FOR BATCHED INFERENCE
class HELMSelfAttention(nn.Module):

    # Initialize the following:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config):
        super().__init__()

        # Grabbing config values from convience
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads
        # support both coupled (d_head = hidden/heads) and decoupled (explicit d_head) configs
        self.d_head = getattr(config, "d_head", None) or (config.hidden_size // config.num_attention_heads)
        self.total_head_dim = self.num_attention_heads * self.d_head
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        self._eval_backend = "dense"
        self._flex_compiled = False
        self._flex_fn = None
        self._block_mask_fn = None


        # QKV Matrix
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias = config.bias
        )

        # RoPE Module
        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta
        )

        # SQK scalers right after RoPE
        self.sqk = nn.Parameter(self.ngpt_sqk_init_scale*torch.ones(self.total_head_dim))

        # Output Matrix
        self.output = nn.Linear(
            self.total_head_dim,
            config.hidden_size,
            bias = config.bias
        )

    # Configure the eval-time attention backend. Call via model.enable_efficient_inference(...).
    #   backend="flex"  : FlexAttention; set compile=True on GPU for the fused kernel (recommended).
    #   backend="gather": compact gather/scatter SDPA, no torch.compile needed.
    #   backend="dense" : compute-all-then-mask (default; what training uses).
    def set_eval_backend(self, backend="flex", compile=True):
        compile = bool(compile)
        # Only drop the cached torch.compile()'d function/block-mask builder when the
        # backend or compile flag actually changes -- resetting on every call (even when
        # nothing changed) forces a full recompilation on the very next forward pass,
        # which is silently expensive if this is called before every timed benchmark run.
        changed = (backend != getattr(self, "_eval_backend", None)
                   or compile != getattr(self, "_flex_compiled", None))
        self._eval_backend = backend
        self._flex_compiled = compile
        if changed:
            self._flex_fn = None
            self._block_mask_fn = None

    def _flex_attn(self, q, k, v, block_mask, scale):
        if self._flex_fn is None:
            from torch.nn.attention.flex_attention import flex_attention
            self._flex_fn = torch.compile(flex_attention) if self._flex_compiled else flex_attention
        return self._flex_fn(q, k, v, block_mask=block_mask, scale=scale)

    def _build_block_mask(self, mask_mod, B, H, S, device):
        if self._block_mask_fn is None:
            from torch.nn.attention.flex_attention import create_block_mask
            # compiling create_block_mask avoids materializing the full SxS mask for long sequences
            self._block_mask_fn = torch.compile(create_block_mask) if self._flex_compiled else create_block_mask
        return self._block_mask_fn(mask_mod, B, H, S, S, device=device)

    # Define Training
    # No router: plain nGPT multi-head attention, ALL heads always active.
    def forward(self, hidden_states, attention_mask):

        # Project hidden_states -> QKV : [b, seq_len, total_head_dim * 3]
        qkv_proj = cast_linear(hidden_states, self.qkv)

        batch_size, seq_len, _ = hidden_states.size()

        # Split into q, k, v : each [b, seq_len, total_head_dim]
        q, k, v = qkv_proj.split(self.total_head_dim, dim=-1)

        # sqk scaler (nGPT) : [total_head_dim] -> [1, num_heads, 1, d_head]
        sqk = (self.sqk * (self.ngpt_sqk_init_value / self.ngpt_sqk_init_scale))
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)

        # Reshape to [b, num_heads, seq_len, d_head]
        q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head).permute(0, 2, 1, 3)
        k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head).permute(0, 2, 1, 3)
        v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head).permute(0, 2, 1, 3)

        # nGPT: unit-norm q,k -> RoPE -> sqk scaling
        q = justnorm(q)
        k = justnorm(k)
        q = self.RoPE(q)
        k = self.RoPE(k)
        q = sqk.to(q.dtype) * q
        k = sqk.to(k.dtype) * k

        # Attention (nGPT scale = sqrt(d_head))
        context_layer = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attention_mask.to(q.dtype),
            scale=math.sqrt(self.d_head),
        )

        # Exclusive attention (kept identical to the routed nGPT model for a clean comparison)
        if self.config.use_exclusive_attention:
            Vn = torch.nn.functional.normalize(v, dim=-1)
            context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

        # (no router_mask, no permanent/elastic split, no jitter -- every head contributes)

        # Reshape back to [b, seq_len, total_head_dim] and project out
        context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()
        context_reshaped = context_reshaped.view(batch_size, seq_len, -1)
        context_layer = cast_linear(context_reshaped, self.output)

        # Return context_layer (normalization occurs in HELMMLP)
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )

    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py

        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv_pre = cast_linear(hidden_states_opt1 ,self.mlp_exp)
        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        # We need to keep suv to be bf16. The line above promoted suc fp32 and the autocaster didn't fix it
        suv = suv.to(uv_pre.dtype)

        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv_post_suv = suv * uv_pre

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention (which defines RotaryEmbeddigs) + HELMMLP
# This is 1 transformer layer
class HELMBlock(nn.Module):

    # No router: just nGPT attention + MLP.
    def __init__(self, config):
        super().__init__()
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)

    # step_tensor / easiness_score accepted but ignored (keeps the trainer call site unchanged).
    def forward(self, hidden_states, attention_mask, step_tensor = None, easiness_score = None):
        attn_output = self.attn(hidden_states, attention_mask)
        layer_output = self.mlp(hidden_states, attn_output)
        # zero aux/sparsity so the model's return contract matches the routed version
        zero = hidden_states.new_zeros(())
        return layer_output, zero, zero



# HELMModel - HELM without the head
class HELMModel(nn.Module):

    # Define the following:
    #   - HELMEmbedding
    #   - HELMBlock
    def __init__(self, config):
        super().__init__()

        # Get the ckpt_attribute
        self.use_ckpt = config.use_ckpt

        # Embedding layer
        self.embedding = HELMEmbedding(config)

        # Transformer blocks
        self.blocks = nn.ModuleList(
            [HELMBlock(config) for _ in range(config.num_hidden_layers)]
        )


    # Forward Pass
    def forward(self, input_ids, attention_mask, current_step = None, easiness_score = None):

        # Build additive mask for SDPA fallback
        # Reshape Additive Mask to be 4D for SDPA [batch_size, 1, 1, seq_len]
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        # Convert the current_step to be infinity if null, or a tensor, or a tensor of the correct datatype if its already tensor
        # We did this because we don't want to pass in a non-tensor if we are using gradient_checkpointing
        if current_step is None:
            step_tensor = torch.tensor(float("inf"), device=input_ids.device)
        elif not isinstance(current_step, torch.Tensor):
            step_tensor = torch.tensor(current_step, device=input_ids.device)
        else:
            step_tensor = current_step

        # Pass input_ids through the input
        embeddings = self.embedding(input_ids)

        # Set Embeddings to be hidden_states
        hidden_states = embeddings.to(torch.bfloat16)

        # Accumulate aux_loss and sparsity_loss
        total_aux_loss = 0
        total_sparsity_loss = 0

        # Run Tranformer Blocks
        for block in self.blocks:
            # Use Gradient Checkpointing
            if self.use_ckpt and self.training:
                _ckpt = (_xla_checkpoint if (_xla_checkpoint is not None
                         and hidden_states.device.type == "xla")
                         else torch.utils.checkpoint.checkpoint)
                hidden_states, aux_loss, sparsity_loss = _ckpt(
                    block,
                    hidden_states,
                    attention_mask,
                    step_tensor,
                    easiness_score,
                    use_reentrant=True if hidden_states.device.type == "xla" else False
                )
            # Or Standard Forward Pass
            else:
                hidden_states, aux_loss, sparsity_loss = block(hidden_states, attention_mask, step_tensor, easiness_score)

            total_aux_loss += aux_loss
            total_sparsity_loss += sparsity_loss

        # Return hidden state (feature extraction / context location prediction) & special losses
        # hidden_states: [b, seq_len, hidden_size]
        return hidden_states, total_aux_loss, total_sparsity_loss



# HELMModelforMaskedLM
class HELMForMaskedLM(PreTrainedModel):

    # Define the Config for the HF push_to_hub() function
    config_class = HELMConfig

    # Define the Following:
    #   - HELMModel
    #   - classifier
    #   - Head layer scaling vector
    def __init__(self, config):
        super().__init__(config)

        # Define from Config
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale

        # Define the Model
        self.model = HELMModel(config)

        # Define the head Layer
        self.classifier = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias = config.bias
        )

        # Define the head layer scaling vetor
        self.sz = nn.Parameter(torch.ones(config.vocab_size))

        # HF Function to call _init_weights() function
        self.post_init()

    # Initialize weights (pulled from ngpt model.py)
    def _init_weights(self, module):

        # If it's an nn.Linear, initialize it with the initializer_range
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        # If it's an nn.Linear, initialize it with the initializer_range also)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    # Define Function to normalize_ngpt_matrices
    # Flip every self-attention layer into an efficient eval backend for GPU inference.
    # Leave this OFF for TPU training/validation (the default "dense" path is static-shape friendly).
    #   model.eval(); model.enable_efficient_inference("flex")        # GPU, fused (recommended)
    #   model.eval(); model.enable_efficient_inference("gather")      # no torch.compile needed
    # On GPU also wrap inference in torch.compile, or pass compile=True (default) to fuse flex.
    def enable_efficient_inference(self, backend="flex", compile=True):
        for block in self.model.blocks:
            block.attn.set_eval_backend(backend=backend, compile=compile)
        return self

    # Define Function to normalize_ngpt_matrices
    def normalize_ngpt_matrices(self):

        # Define all the projection matrices to normalize
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
        )

        # Normalize every one of those mats along their dim = 1 (embedding)
        # The model's weights are transposed (for backprop) so instead of dim = 0, we do dim = 1
        with torch.no_grad():
            for name, param in self.named_parameters():
                if name.endswith(keys_to_normalize):
                    # EDIT: Instead of complete data-reassignment (danger-danger!!!), use in_place copying
                    param.copy_(justnorm(param, dim = 1, eps = 1e-12))

    # Get all necessary telemetrics & return as dict
    @torch.no_grad()
    def get_telemetry(self):

        # Define Telemetry:
        telemetry = {}

        # Iterate Through All Layers
        for i, block in enumerate(self.model.blocks):

            # (no router in this baseline -- no sigmoid_scores / flat_mask / tau / l_i_weights)

            # ---------- SELF ATTENTION ----------

            # sqk vector (scaling vector in attn, element wise mult.)
            sqk_tensor = block.attn.sqk.detach().cpu()
            telemetry[f"layer_{i}_sqk_mean"] = sqk_tensor.mean().item()
            telemetry[f"layer_{i}_sqk_std"] = sqk_tensor.std().item()
            telemetry[f"layer_{i}_sqk_hist"] = sqk_tensor

            # ------------------------------------


            # @@@@@@@@@@ MLP @@@@@@@@@@

            # Get MLP block
            mlp = block.mlp

            # attn_alpha (post attention eigen learning rates)
            attn_alpha_tensor = mlp.attn_alpha.detach().cpu()
            telemetry[f"layer_{i}_attn_alpha_mean"] = attn_alpha_tensor.mean().item()
            telemetry[f"layer_{i}_attn_alpha_std"] = attn_alpha_tensor.std().item()
            telemetry[f"layer_{i}_attn_alpha_hist"] = attn_alpha_tensor

            # mlp_alpha (post mlp eigen learning rates)
            mlp_alpha_tensor = mlp.mlp_alpha.detach().cpu()
            telemetry[f"layer_{i}_mlp_alpha_mean"] = mlp_alpha_tensor.mean().item()
            telemetry[f"layer_{i}_mlp_alpha_std"] = mlp_alpha_tensor.std().item()
            telemetry[f"layer_{i}_mlp_alpha_hist"] = mlp_alpha_tensor

            # suv scaling vector ([intermediate_size * 2] scaling vector in u and v, element wise mult.)
            suv_tensor = mlp.suv.detach().cpu()
            telemetry[f"layer_{i}_suv_mean"] = suv_tensor.mean().item()
            telemetry[f"layer_{i}_suv_std"] = suv_tensor.std().item()
            telemetry[f"layer_{i}_suv_hist"] = suv_tensor

            # @@@@@@@@@@@@@@@@@@@@@@@@@


        # sz_tensor ([intermediate_size * 2] scaling vector in u and v, element wise mult.)
        sz_tensor = self.sz.detach().cpu()
        telemetry["lm_head_sz_mean"] = sz_tensor.mean().item()
        telemetry["lm_head_sz_std"] = sz_tensor.std().item()
        telemetry["lm_head_sz_hist"] = sz_tensor

        return telemetry



    # Forward pass
    def forward(self, input_ids, attention_mask, current_step = None, easiness_score = None):

        # Gather Context from the model
        # features: [b, seq_len, hidden_size]
        features, total_aux_loss, total_sparsity_loss = self.model(input_ids, attention_mask, current_step, easiness_score)

        # Scale / prepare sz
        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)

        # project features onto classifer
        # [b, seq_len, hidden_size] * [hidden_size, vocab_size] = [b, seq_len, vocab_size]
        unscaled_logits = cast_linear(features, self.classifier)

        # Scale the logits with sz
        logits = sz.to(unscaled_logits.dtype) * unscaled_logits

        # Return Logits
        return logits, total_aux_loss, total_sparsity_loss

Writing model.py


## 2. Write the single-checkpoint audit

Key distinction: the script reports both **raw effective rank** (magnitude-sensitive) and **directional effective rank** (each head normalized first). This prevents us from calling unequal-strength but distinct heads "redundant."


In [8]:
%%writefile audit_dense16.py
#!/usr/bin/env python3
"""
Dense-16 HELM / vanilla nGPT assumption audit.

Purpose
-------
This is deliberately NOT a router diagnostic. It audits the plain 16-head dense
baseline before we make further assumptions about what a "healthy" head bank
should look like.

Questions tested
----------------
1. Does a dense 16-head model actually have high head-level effective rank?
2. Is low raw effective rank caused by duplicate directions or unequal energy?
3. Does redundancy already exist in pre-output attention contexts, or does it
   appear mainly after each head passes through its W_O block?
4. Are all 16 heads useful to MLM CE, or can individual heads be removed cheaply?
5. Does residual RMS / energy predict actual single-head ablation importance?
6. How gracefully does a model TRAINED dense degrade if we keep only K heads
   per layer at evaluation time?
7. Does "keep the largest-energy heads" outperform random post-hoc subsets?

Important interpretation rule
-----------------------------
This script measures several kinds of "rank" separately:

- raw effective rank: magnitude-sensitive; falls if a few heads dominate energy
- directional effective rank: computed after normalizing every head; measures
  directional redundancy without punishing unequal head magnitude

A low raw rank with a high directional rank means "different heads, unequal
strength", NOT "duplicate heads".

The head-removal curve is a POST-HOC pruning diagnostic. It does not replace a
separately trained smaller dense model.
"""

from __future__ import annotations

import argparse
import contextlib
import csv
import glob
import json
import math
import os
import random
import re
import shutil
import sys
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

try:
    import pyarrow.parquet as pq
except Exception as exc:
    raise RuntimeError("pyarrow is required: pip install pyarrow") from exc

try:
    from huggingface_hub import HfApi, hf_hub_download
except Exception as exc:
    raise RuntimeError("huggingface_hub is required") from exc

SCRIPT_DIR = Path.cwd()
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

try:
    from model import HELMConfig, HELMForMaskedLM, justnorm, cast_linear
except Exception as exc:
    raise RuntimeError(
        "Could not import model.py. Run the vanilla nGPT model cell first so "
        "the uploaded dense-16 architecture is written to model.py."
    ) from exc


# -----------------------------------------------------------------------------
# Defaults from the uploaded vanilla model / current HELM validation setup
# -----------------------------------------------------------------------------
MODEL_REPO = "JamesResearch1216/phase06v4-Vanilla"
CHECKPOINT_FILE = "latest"
DATA_REPO = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6"
VALIDATION_FILE = "data/seq_1024/validation-00000.parquet"


# -----------------------------------------------------------------------------
# Generic helpers
# -----------------------------------------------------------------------------
def get_hf_token() -> Optional[str]:
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def strip_state_prefixes(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for key, value in state.items():
        k = key
        changed = True
        while changed:
            changed = False
            for prefix in ("module.", "_orig_mod."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
                    changed = True
        out[k] = value
    return out


def rankdata_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=np.float64)
    sorted_x = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sorted_x[j] == sorted_x[i]:
            j += 1
        ranks[order[i:j]] = 0.5 * (i + j - 1)
        i = j
    return ranks


def spearman_np(x, y) -> float:
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    good = np.isfinite(x) & np.isfinite(y)
    x, y = x[good], y[good]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan")
    return float(np.corrcoef(rankdata_np(x), rankdata_np(y))[0, 1])


def effective_rank_from_gram(gram: np.ndarray) -> Tuple[float, float, np.ndarray]:
    g = 0.5 * (gram + gram.T)
    vals = np.linalg.eigvalsh(g)
    vals = np.clip(vals, 0.0, None)
    total = vals.sum()
    if total <= 1e-12:
        return 0.0, 0.0, vals
    p = vals / total
    p_pos = p[p > 1e-12]
    entropy_rank = float(np.exp(-(p_pos * np.log(p_pos)).sum()))
    participation_rank = float((total * total) / (np.square(vals).sum() + 1e-12))
    return entropy_rank, participation_rank, vals


def cosine_from_gram(gram: np.ndarray) -> np.ndarray:
    diag = np.clip(np.diag(gram), 0.0, None)
    denom = np.sqrt(np.outer(diag, diag)) + 1e-12
    cos = gram / denom
    cos = np.clip(cos, -1.0, 1.0)
    return 0.5 * (cos + cos.T)


def offdiag_stats(cos: np.ndarray) -> Dict[str, float]:
    h = cos.shape[0]
    vals = cos[~np.eye(h, dtype=bool)]
    av = np.abs(vals)
    return {
        "mean_abs": float(av.mean()) if av.size else 0.0,
        "median_abs": float(np.median(av)) if av.size else 0.0,
        "max_abs": float(av.max()) if av.size else 0.0,
        "frac_abs_gt_0.5": float((av > 0.5).mean()) if av.size else 0.0,
        "frac_abs_gt_0.8": float((av > 0.8).mean()) if av.size else 0.0,
    }


def components_for_fraction(eigvals: np.ndarray, fraction: float) -> int:
    vals = np.sort(np.clip(eigvals, 0.0, None))[::-1]
    total = vals.sum()
    if total <= 1e-12:
        return 0
    c = np.cumsum(vals) / total
    return int(np.searchsorted(c, fraction, side="left") + 1)


def gini_nonnegative(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=np.float64).reshape(-1)
    x = np.clip(x, 0.0, None)
    if x.size == 0 or x.sum() <= 1e-12:
        return 0.0
    x = np.sort(x)
    n = x.size
    idx = np.arange(1, n + 1, dtype=np.float64)
    return float((2.0 * np.sum(idx * x) / (n * x.sum())) - (n + 1.0) / n)


def save_heatmap(path: Path, matrix: np.ndarray, title: str, vmin=None, vmax=None) -> None:
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(matrix, aspect="auto", interpolation="nearest", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("Head")
    ax.set_ylabel("Head")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_bar(path: Path, values: np.ndarray, title: str, ylabel: str) -> None:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(np.arange(len(values)), values)
    ax.set_title(title)
    ax.set_xlabel("Head")
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_spectrum(path: Path, eigvals: np.ndarray, title: str) -> None:
    vals = np.sort(np.clip(eigvals, 0.0, None))[::-1]
    frac = vals / (vals.sum() + 1e-12)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.arange(1, len(frac) + 1), frac, marker="o")
    ax.set_title(title)
    ax.set_xlabel("Component")
    ax.set_ylabel("Fraction of energy")
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


# -----------------------------------------------------------------------------
# Device
# -----------------------------------------------------------------------------
@dataclass
class DeviceContext:
    device: torch.device
    kind: str
    xm: object = None
    autocast_dtype: Optional[torch.dtype] = None

    def autocast(self):
        if self.kind == "cuda":
            return torch.autocast(device_type="cuda", dtype=self.autocast_dtype)
        if self.kind == "xla":
            try:
                return torch.autocast(device_type="xla", dtype=torch.bfloat16)
            except Exception:
                return contextlib.nullcontext()
        return contextlib.nullcontext()

    def mark_step(self):
        if self.kind == "xla" and self.xm is not None:
            self.xm.mark_step()


def resolve_device(requested: str) -> DeviceContext:
    requested = requested.lower()
    if requested == "auto":
        if torch.cuda.is_available():
            requested = "cuda"
        elif glob.glob("/dev/accel*"):
            requested = "xla"
        else:
            requested = "cpu"

    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable")
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        return DeviceContext(torch.device("cuda"), "cuda", autocast_dtype=dtype)

    if requested == "xla":
        try:
            import torch_xla.core.xla_model as xm
        except Exception as exc:
            raise RuntimeError("XLA requested but torch_xla could not be imported") from exc
        return DeviceContext(xm.xla_device(), "xla", xm=xm, autocast_dtype=torch.bfloat16)

    if requested == "cpu":
        return DeviceContext(torch.device("cpu"), "cpu")

    raise ValueError(f"Unknown device: {requested}")


# -----------------------------------------------------------------------------
# Assets / checkpoint resolution
# -----------------------------------------------------------------------------
def resolve_checkpoint_name(repo_id: str, requested: str, token: Optional[str]) -> str:
    if requested.lower() != "latest":
        return requested

    api = HfApi(token=token)
    files = api.list_repo_files(repo_id=repo_id, repo_type="model", token=token)
    candidates = []
    for name in files:
        m = re.search(r"checkpoint-(\d+)\.pt$", name)
        if m:
            candidates.append((int(m.group(1)), name))

    if not candidates:
        raise RuntimeError(
            f"No checkpoint-XXXXXX.pt files found in {repo_id}. "
            "Pass --checkpoint explicitly."
        )

    step, name = max(candidates)
    print(f"Auto-selected latest checkpoint: {name} (step {step})")
    return name


def checkpoint_step_from_name(name: str) -> int:
    m = re.search(r"checkpoint-(\d+)\.pt$", name)
    return int(m.group(1)) if m else 0


def download_assets(cache_dir: Path, token: Optional[str], model_repo: str,
                    checkpoint_file: str, data_repo: str, validation_file: str):
    cache_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_file = resolve_checkpoint_name(model_repo, checkpoint_file, token)

    print(f"Downloading checkpoint: {model_repo}/{checkpoint_file}")
    ckpt_path = hf_hub_download(
        repo_id=model_repo,
        filename=checkpoint_file,
        repo_type="model",
        token=token,
        local_dir=str(cache_dir / "model_repo"),
    )

    print(f"Downloading validation shard: {data_repo}/{validation_file}")
    validation_path = hf_hub_download(
        repo_id=data_repo,
        filename=validation_file,
        repo_type="dataset",
        token=token,
        local_dir=str(cache_dir / "dataset"),
    )

    return Path(ckpt_path), Path(validation_path), checkpoint_file


def load_model(ckpt_path: Path, dev: DeviceContext):
    config = HELMConfig()
    model = HELMForMaskedLM(config)

    print(f"Architecture from model.py: H={config.num_attention_heads}, "
          f"hidden={config.hidden_size}, d_head={getattr(config, 'd_head', None)}")

    payload = torch.load(str(ckpt_path), map_location="cpu")
    if not isinstance(payload, dict):
        raise RuntimeError("Checkpoint is not a dict")
    state = strip_state_prefixes(payload.get("model_state", payload))
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f"State load: {len(missing)} missing, {len(unexpected)} unexpected")
        if missing:
            print("  missing first 10:", missing[:10])
        if unexpected:
            print("  unexpected first 10:", unexpected[:10])
        if len(missing) > 5 or len(unexpected) > 5:
            raise RuntimeError("Large state-dict mismatch: model.py likely does not match checkpoint")

    del payload
    model.to(dev.device)
    model.eval()
    return model, config


# -----------------------------------------------------------------------------
# Validation masking / CE
# -----------------------------------------------------------------------------
def deterministic_span_mask(ids: torch.Tensor, config: HELMConfig, seed: int,
                            probability: float = 0.30, span_length: int = 3):
    ids = ids.clone().long()
    labels = torch.full_like(ids, -100)
    g = torch.Generator(device="cpu")
    g.manual_seed(int(seed))

    special = {
        int(config.bos_token_id), int(config.eos_token_id), int(config.pad_token_id),
        int(config.mask_token_id), int(config.unk_token_id),
    }
    candidate = [i for i, tok in enumerate(ids.tolist()) if int(tok) not in special]
    if not candidate:
        return ids, labels

    target = max(1, int(round(probability * len(candidate))))
    candidate_set = set(candidate)
    perm = torch.randperm(len(candidate), generator=g).tolist()
    chosen = set()
    for pi in perm:
        if len(chosen) >= target:
            break
        start = candidate[pi]
        for pos in range(start, min(start + span_length, ids.numel())):
            if pos in candidate_set:
                chosen.add(pos)
                if len(chosen) >= target:
                    break
    chosen = sorted(chosen) or [candidate[0]]
    pos = torch.tensor(chosen, dtype=torch.long)
    labels[pos] = ids[pos].clone()

    r = torch.rand(len(pos), generator=g)
    mask_sel = r < 0.80
    random_sel = (r >= 0.80) & (r < 0.90)
    ids[pos[mask_sel]] = int(config.mask_token_id)
    if random_sel.any():
        random_tokens = torch.randint(0, int(config.vocab_size),
                                      (int(random_sel.sum()),), generator=g)
        ids[pos[random_sel]] = random_tokens
    return ids, labels


def prepare_batches(validation_path: Path, config: HELMConfig, num_examples: int,
                    batch_size: int, seq_len: int, seed: int):
    table = pq.read_table(str(validation_path), columns=["input_ids"])
    total_rows = table.num_rows
    n = min(num_examples, total_rows)
    rng = np.random.default_rng(seed)
    indices = rng.permutation(total_rows)[:n]
    input_col = table.column("input_ids")

    examples = []
    for i, row_idx in enumerate(indices.tolist()):
        ids = torch.tensor(input_col[row_idx].as_py(), dtype=torch.long)[:seq_len]
        if ids.numel() < seq_len:
            ids = torch.cat([
                ids,
                torch.full((seq_len - ids.numel(),), int(config.pad_token_id), dtype=torch.long),
            ])
        masked, labels = deterministic_span_mask(ids, config, seed + 100003 * i)
        examples.append({
            "input_ids": masked,
            "labels": labels,
            "attention_mask": (ids != int(config.pad_token_id)).long(),
        })

    usable = (len(examples) // batch_size) * batch_size
    examples = examples[:usable]
    if not examples:
        raise RuntimeError("Not enough examples for one batch")

    batches = []
    for start in range(0, len(examples), batch_size):
        chunk = examples[start:start + batch_size]
        batches.append({k: torch.stack([x[k] for x in chunk]) for k in chunk[0]})

    print(f"Prepared {len(examples)} examples -> {len(batches)} batches, seq_len={seq_len}")
    return batches


def move_batch(batch, dev: DeviceContext):
    return {k: v.to(dev.device) for k, v in batch.items()}


def ce_sum_and_count(logits: torch.Tensor, labels: torch.Tensor, chunk_tokens: int = 128):
    total = logits.new_zeros((), dtype=torch.float32)
    count = labels.new_zeros((), dtype=torch.long)
    seq_len, vocab = logits.size(1), logits.size(-1)
    for start in range(0, seq_len, chunk_tokens):
        end = min(start + chunk_tokens, seq_len)
        lgt = logits[:, start:end, :].float().reshape(-1, vocab)
        lab = labels[:, start:end].reshape(-1)
        total = total + F.cross_entropy(lgt, lab, ignore_index=-100, reduction="sum")
        count = count + (lab != -100).sum()
    return total, count


def forward_ce(model, batch, dev: DeviceContext, checkpoint_step: int):
    with dev.autocast():
        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            current_step=checkpoint_step,
        )
        logits = out[0] if isinstance(out, (tuple, list)) else out
        total, count = ce_sum_and_count(logits, batch["labels"])
    return total, count


def evaluate_ce(model, batches, dev: DeviceContext, checkpoint_step: int, max_batches=None):
    total_sum = 0.0
    total_count = 0
    subset = batches if max_batches is None else batches[:max_batches]
    with torch.no_grad():
        for cpu_batch in subset:
            batch = move_batch(cpu_batch, dev)
            s, c = forward_ce(model, batch, dev, checkpoint_step)
            dev.mark_step()
            total_sum += float(s.detach().cpu())
            total_count += int(c.detach().cpu())
    return total_sum / max(1, total_count)


# -----------------------------------------------------------------------------
# Capture layer inputs and recompute exact pre-output contexts
# -----------------------------------------------------------------------------
class AttentionInputCapture:
    def __init__(self, model, layers: Sequence[int]):
        self.model = model
        self.layers = set(layers)
        self.handles = []
        self.data: Dict[int, Tuple[torch.Tensor, torch.Tensor]] = {}

    def _hook(self, li: int):
        def hook(module, inputs):
            hidden_states = inputs[0]
            attention_mask = inputs[1]
            self.data[li] = (hidden_states.detach(), attention_mask.detach())
        return hook

    def __enter__(self):
        for li in self.layers:
            self.handles.append(
                self.model.model.blocks[li].attn.register_forward_pre_hook(self._hook(li))
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


def unmasked_attention_context(attn, hidden_states: torch.Tensor, attention_mask: torch.Tensor):
    qkv_proj = cast_linear(hidden_states, attn.qkv)
    b, s, _ = hidden_states.shape
    q, k, v = qkv_proj.split(attn.total_head_dim, dim=-1)
    q = q.view(b, s, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    k = k.view(b, s, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    v = v.view(b, s, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)

    q = justnorm(q)
    k = justnorm(k)
    q = attn.RoPE(q)
    k = attn.RoPE(k)

    sqk = attn.sqk * (attn.ngpt_sqk_init_value / attn.ngpt_sqk_init_scale)
    sqk = sqk.view(1, attn.num_attention_heads, 1, attn.d_head).to(q.dtype)
    q = sqk * q
    k = sqk * k

    context = F.scaled_dot_product_attention(
        q, k, v,
        attn_mask=attention_mask.to(q.dtype),
        scale=math.sqrt(attn.d_head),
    )

    if attn.config.use_exclusive_attention:
        vn = F.normalize(v, dim=-1)
        context = context - (context * vn).sum(dim=-1, keepdim=True) * vn
    return context


# -----------------------------------------------------------------------------
# Dense head-bank audit
# -----------------------------------------------------------------------------
def head_bank_audit(model, batches, dev: DeviceContext, checkpoint_step: int,
                    max_batches: int, sample_tokens: int, output_dir: Path,
                    layers: Optional[Sequence[int]] = None):
    if layers is None:
        layers = list(range(len(model.model.blocks)))
    else:
        layers = list(layers)
    H = model.config.num_attention_heads

    context_grams = {li: np.zeros((H, H), dtype=np.float64) for li in layers}
    residual_grams = {li: np.zeros((H, H), dtype=np.float64) for li in layers}
    context_counts = {li: 0 for li in layers}
    residual_counts = {li: 0 for li in layers}

    with torch.no_grad():
        for cpu_batch in batches[:max_batches]:
            batch = move_batch(cpu_batch, dev)
            with AttentionInputCapture(model, layers) as cap:
                with dev.autocast():
                    _ = model(
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                        current_step=checkpoint_step,
                    )
                dev.mark_step()

            for li in layers:
                hidden, attn_mask = cap.data[li]
                attn = model.model.blocks[li].attn
                with dev.autocast():
                    context = unmasked_attention_context(attn, hidden, attn_mask)  # [B,H,S,d]
                    seq = context.size(2)
                    t = min(sample_tokens, seq)
                    positions = torch.linspace(0, seq - 1, steps=t, device=context.device).long()
                    c = context.index_select(2, positions)  # [B,H,T,d]

                    # Head context vectors BEFORE W_O.
                    cflat = c.permute(1, 0, 2, 3).contiguous().view(H, -1).float()
                    cgram = cflat @ cflat.T

                    # Head residual contributions AFTER each W_O block.
                    W = attn.output.weight.to(c.dtype).view(attn.hidden_size, H, attn.d_head)
                    y = torch.einsum("bhtd,ohd->bhto", c, W)  # [B,H,T,D]
                    yflat = y.permute(1, 0, 2, 3).contiguous().view(H, -1).float()
                    ygram = yflat @ yflat.T

                dev.mark_step()
                context_grams[li] += (cgram.detach().to(torch.float32).cpu().numpy().astype(np.float64))
                residual_grams[li] += (ygram.detach().to(torch.float32).cpu().numpy().astype(np.float64))
                context_counts[li] += int(cflat.shape[1])
                residual_counts[li] += int(yflat.shape[1])
                del context, c, cflat, cgram, y, yflat, ygram

    results = {}
    all_rows = []

    for li in layers:
        attn = model.model.blocks[li].attn
        cg = context_grams[li]
        rg = residual_grams[li]
        ccos = cosine_from_gram(cg)
        rcos = cosine_from_gram(rg)

        c_raw_er, c_raw_pr, c_raw_eig = effective_rank_from_gram(cg)
        r_raw_er, r_raw_pr, r_raw_eig = effective_rank_from_gram(rg)
        # Directional rank: every head normalized to unit norm first.
        c_dir_er, c_dir_pr, c_dir_eig = effective_rank_from_gram(ccos)
        r_dir_er, r_dir_pr, r_dir_eig = effective_rank_from_gram(rcos)

        context_energy = np.clip(np.diag(cg), 0.0, None)
        residual_energy = np.clip(np.diag(rg), 0.0, None)
        context_rms = np.sqrt(context_energy / max(1, context_counts[li]))
        residual_rms = np.sqrt(residual_energy / max(1, residual_counts[li]))
        residual_frac = residual_energy / (residual_energy.sum() + 1e-12)

        # W_O block norm per head: one [hidden_size, d_head] slice per head.
        w = attn.output.weight.detach().float().cpu().numpy()
        w = w.reshape(attn.hidden_size, H, attn.d_head)
        w_block_frob = np.sqrt(np.square(w).sum(axis=(0, 2)))

        # Q/K/V block norm per head for a secondary parameter-allocation check.
        qkv = attn.qkv.weight.detach().float().cpu().numpy()  # [3*H*d, D]
        qkv = qkv.reshape(3, H, attn.d_head, attn.hidden_size)
        qkv_block_frob = np.sqrt(np.square(qkv).sum(axis=(0, 2, 3)))

        rstats = offdiag_stats(rcos)
        cstats = offdiag_stats(ccos)
        coherence = float(rg.sum() / (np.trace(rg) + 1e-12))

        results[li] = {
            "context_raw_entropy_rank": c_raw_er,
            "context_raw_participation_rank": c_raw_pr,
            "context_directional_entropy_rank": c_dir_er,
            "context_directional_participation_rank": c_dir_pr,
            "residual_raw_entropy_rank": r_raw_er,
            "residual_raw_participation_rank": r_raw_pr,
            "residual_directional_entropy_rank": r_dir_er,
            "residual_directional_participation_rank": r_dir_pr,
            "context_cosine": cstats,
            "residual_cosine": rstats,
            "residual_top1_energy_fraction": float(np.max(residual_frac)),
            "residual_top4_energy_fraction": float(np.sort(residual_frac)[-min(4,H):].sum()),
            "residual_energy_gini": gini_nonnegative(residual_energy),
            "residual_components_90pct": components_for_fraction(r_raw_eig, 0.90),
            "residual_components_95pct": components_for_fraction(r_raw_eig, 0.95),
            "head_coherence_ratio": coherence,
            "context_rms": context_rms,
            "residual_rms": residual_rms,
            "residual_energy_fraction": residual_frac,
            "output_block_frobenius": w_block_frob,
            "qkv_block_frobenius": qkv_block_frob,
            "context_rms_vs_residual_rms_spearman": spearman_np(context_rms, residual_rms),
            "wo_norm_vs_residual_rms_spearman": spearman_np(w_block_frob, residual_rms),
        }

        rows = []
        for h in range(H):
            row = {
                "layer": li,
                "head": h,
                "context_rms": float(context_rms[h]),
                "residual_rms": float(residual_rms[h]),
                "residual_energy_fraction": float(residual_frac[h]),
                "output_block_frobenius": float(w_block_frob[h]),
                "qkv_block_frobenius": float(qkv_block_frob[h]),
            }
            rows.append(row)
            all_rows.append(row)

        with (output_dir / f"layer_{li:02d}_head_metrics.csv").open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader(); writer.writerows(rows)

        save_heatmap(output_dir / f"layer_{li:02d}_context_cosine.png", ccos,
                     f"Layer {li}: pre-W_O head context cosine", -1.0, 1.0)
        save_heatmap(output_dir / f"layer_{li:02d}_residual_cosine.png", rcos,
                     f"Layer {li}: residual contribution cosine", -1.0, 1.0)
        save_bar(output_dir / f"layer_{li:02d}_residual_rms.png", residual_rms,
                 f"Layer {li}: per-head residual RMS", "Residual RMS")
        save_bar(output_dir / f"layer_{li:02d}_energy_fraction.png", residual_frac,
                 f"Layer {li}: residual energy fraction", "Fraction")
        save_spectrum(output_dir / f"layer_{li:02d}_raw_rank_spectrum.png", r_raw_eig,
                      f"Layer {li}: raw residual head spectrum")
        save_spectrum(output_dir / f"layer_{li:02d}_directional_rank_spectrum.png", r_dir_eig,
                      f"Layer {li}: normalized-direction head spectrum")

    with (output_dir / "head_metrics_all_layers.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=all_rows[0].keys())
        writer.writeheader(); writer.writerows(all_rows)

    return results


# -----------------------------------------------------------------------------
# Exact head removal by zeroing that head's W_O columns
# -----------------------------------------------------------------------------
@contextmanager
def temporarily_zero_output_heads(model, dropped_by_layer: Dict[int, Sequence[int]]):
    backups = {}
    try:
        with torch.no_grad():
            for li, heads in dropped_by_layer.items():
                attn = model.model.blocks[li].attn
                cols = []
                for h in heads:
                    start = int(h) * attn.d_head
                    cols.extend(range(start, start + attn.d_head))
                if not cols:
                    continue
                idx = torch.tensor(cols, device=attn.output.weight.device, dtype=torch.long)
                backups[li] = (idx, attn.output.weight.index_select(1, idx).clone())
                attn.output.weight[:, idx] = 0
        yield
    finally:
        with torch.no_grad():
            for li, (idx, backup) in backups.items():
                model.model.blocks[li].attn.output.weight[:, idx] = backup


def parse_layers(spec: str, n_layers: int) -> List[int]:
    if spec.lower() == "all":
        return list(range(n_layers))
    out = []
    for part in spec.split(","):
        part = part.strip()
        if part:
            li = int(part)
            if not 0 <= li < n_layers:
                raise ValueError(f"Layer {li} out of range")
            out.append(li)
    return sorted(set(out))


def exact_single_head_ablation(model, batches, dev: DeviceContext, checkpoint_step: int,
                               layers: Sequence[int], max_batches: int,
                               head_metrics: Dict[int, dict], output_dir: Path):
    subset = batches[:max_batches]
    dense_ce = evaluate_ce(model, subset, dev, checkpoint_step)
    H = model.config.num_attention_heads
    rows = []
    summaries = {}

    for li in layers:
        for h in range(H):
            with temporarily_zero_output_heads(model, {li: [h]}):
                ce = evaluate_ce(model, subset, dev, checkpoint_step)
            delta = ce - dense_ce
            rows.append({
                "layer": li,
                "head": h,
                "dense_ce": dense_ce,
                "ablated_ce": ce,
                "delta_ce": delta,
                "residual_rms": float(head_metrics[li]["residual_rms"][h]),
                "energy_fraction": float(head_metrics[li]["residual_energy_fraction"][h]),
            })
            print(f"Exact ablation L{li:02d} H{h:02d}: ΔCE={delta:+.6f}")

        rr = [r for r in rows if r["layer"] == li]
        deltas = np.array([r["delta_ce"] for r in rr], dtype=np.float64)
        rms = np.array([r["residual_rms"] for r in rr], dtype=np.float64)
        energy = np.array([r["energy_fraction"] for r in rr], dtype=np.float64)
        summaries[li] = {
            "mean_delta_ce": float(deltas.mean()),
            "median_delta_ce": float(np.median(deltas)),
            "max_delta_ce": float(deltas.max()),
            "min_delta_ce": float(deltas.min()),
            "fraction_delta_gt_0": float((deltas > 0).mean()),
            "fraction_abs_delta_lt_0.001": float((np.abs(deltas) < 0.001).mean()),
            "rms_vs_delta_ce_spearman": spearman_np(rms, deltas),
            "energy_vs_delta_ce_spearman": spearman_np(energy, deltas),
        }

    with (output_dir / "exact_single_head_ablation.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader(); writer.writerows(rows)

    return {"dense_ce_subset": dense_ce, "per_layer": summaries, "rows": rows}


# -----------------------------------------------------------------------------
# Post-hoc head-count curve on the dense-trained model
# -----------------------------------------------------------------------------
def evaluate_posthoc_head_curve(model, batches, dev: DeviceContext, checkpoint_step: int,
                                head_metrics: Dict[int, dict], keep_counts: Sequence[int],
                                trials: int, max_batches: int, seed: int, output_dir: Path):
    subset = batches[:max_batches]
    H = model.config.num_attention_heads
    L = len(model.model.blocks)
    rng = np.random.default_rng(seed + 555)
    rows = []

    dense_ce = evaluate_ce(model, subset, dev, checkpoint_step)
    rows.append({"method": "dense", "keep_count": H, "trial": 0, "ce": dense_ce})

    for k in sorted(set(keep_counts)):
        if k < 1 or k > H:
            continue
        if k == H:
            continue

        # Magnitude heuristic: keep the K largest residual-energy heads independently per layer.
        dropped = {}
        for li in range(L):
            energy = np.asarray(head_metrics[li]["residual_energy_fraction"])
            keep = set(np.argsort(-energy)[:k].tolist())
            dropped[li] = [h for h in range(H) if h not in keep]
        with temporarily_zero_output_heads(model, dropped):
            ce = evaluate_ce(model, subset, dev, checkpoint_step)
        rows.append({"method": "top_energy", "keep_count": k, "trial": 0, "ce": ce})
        print(f"Post-hoc keep {k:02d}/16 top-energy: CE={ce:.6f}")

        # Random same-count subsets, independently sampled per layer.
        for t in range(trials):
            dropped = {}
            for li in range(L):
                keep = set(rng.choice(H, size=k, replace=False).tolist())
                dropped[li] = [h for h in range(H) if h not in keep]
            with temporarily_zero_output_heads(model, dropped):
                ce = evaluate_ce(model, subset, dev, checkpoint_step)
            rows.append({"method": "random", "keep_count": k, "trial": t, "ce": ce})
            print(f"Post-hoc keep {k:02d}/16 random trial {t}: CE={ce:.6f}")

    with (output_dir / "posthoc_head_count_curve.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader(); writer.writerows(rows)

    # Aggregate for summary and plot.
    agg = {}
    for k in sorted(set(r["keep_count"] for r in rows)):
        agg[k] = {}
        for method in sorted(set(r["method"] for r in rows if r["keep_count"] == k)):
            vals = np.array([r["ce"] for r in rows if r["keep_count"] == k and r["method"] == method])
            agg[k][method] = {"mean": float(vals.mean()), "std": float(vals.std())}

    fig, ax = plt.subplots(figsize=(8, 4))
    for method in ("dense", "top_energy", "random"):
        xs, ys = [], []
        for k in sorted(agg):
            if method in agg[k]:
                xs.append(k); ys.append(agg[k][method]["mean"])
        if xs:
            ax.plot(xs, ys, marker="o", label=method)
    ax.set_xlabel("Heads kept per layer")
    ax.set_ylabel("Masked-LM CE")
    ax.set_title("Post-hoc head-count curve (dense-trained checkpoint)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "posthoc_head_count_curve.png", dpi=160)
    plt.close(fig)

    return {"dense_ce_subset": dense_ce, "aggregate": agg, "rows": rows}


# -----------------------------------------------------------------------------
# Reporting
# -----------------------------------------------------------------------------
def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, float) and not np.isfinite(obj):
        return None
    return obj


def write_summary(output_dir: Path, args, checkpoint_name: str, config,
                  baseline_ce: float, head_metrics, exact, curve):
    H = config.num_attention_heads
    d_head = getattr(config, "d_head", None) or (config.hidden_size // H)
    lines = []
    lines.append("# Dense-16 HELM Assumption Audit\n")
    lines.append(f"Checkpoint: `{args.model_repo}/{checkpoint_name}`\n")
    lines.append(f"Architecture: {H} heads × {d_head} dims = {H*d_head} attention width; hidden={config.hidden_size}.\n")
    lines.append(f"Diagnostic MLM CE: **{baseline_ce:.6f}**\n")

    lines.append("\n## 1. Effective rank and head geometry\n")
    lines.append("Raw rank is magnitude-sensitive. Directional rank normalizes every head first, so it measures directional diversity without forcing equal head strength.\n")
    lines.append("\n|Layer|Context raw|Context directional|Residual raw|Residual directional|Mean |cos| residual|Top-1 energy|Top-4 energy|90% components|\n")
    lines.append("|---:|---:|---:|---:|---:|---:|---:|---:|---:|\n")
    for li in sorted(head_metrics):
        r = head_metrics[li]
        lines.append(
            f"|{li}|{r['context_raw_entropy_rank']:.2f}/{H}|"
            f"{r['context_directional_entropy_rank']:.2f}/{H}|"
            f"{r['residual_raw_entropy_rank']:.2f}/{H}|"
            f"{r['residual_directional_entropy_rank']:.2f}/{H}|"
            f"{r['residual_cosine']['mean_abs']:.3f}|"
            f"{100*r['residual_top1_energy_fraction']:.1f}%|"
            f"{100*r['residual_top4_energy_fraction']:.1f}%|"
            f"{r['residual_components_90pct']}|\n"
        )

    lines.append("\n### How to read the rank split\n")
    lines.append("- **low raw + high directional rank** → heads are mostly different directions but energy is concentrated in a few strong heads.\n")
    lines.append("- **low raw + low directional rank** → genuine directional redundancy/collapse is also present.\n")
    lines.append("- **context rank high, residual rank low** → the attention heads themselves are diverse but W_O compresses/concentrates their residual contributions.\n")
    lines.append("- **context and residual ranks both low** → redundancy already exists before W_O.\n")

    lines.append("\n## 2. Parameter allocation vs functional strength\n")
    for li in sorted(head_metrics):
        r = head_metrics[li]
        lines.append(
            f"- L{li:02d}: rho(context RMS, residual RMS)={r['context_rms_vs_residual_rms_spearman']:.3f}; "
            f"rho(W_O block norm, residual RMS)={r['wo_norm_vs_residual_rms_spearman']:.3f}; "
            f"energy Gini={r['residual_energy_gini']:.3f}; coherence={r['head_coherence_ratio']:.3f}.\n"
        )

    lines.append("\n## 3. Exact single-head ablation\n")
    if exact is None:
        lines.append("Skipped. Re-run with `--exact-ablation`.\n")
    else:
        lines.append(f"Dense CE on ablation subset: **{exact['dense_ce_subset']:.6f}**\n")
        for li, r in exact["per_layer"].items():
            lines.append(
                f"- L{int(li):02d}: mean ΔCE={r['mean_delta_ce']:+.6f}, max={r['max_delta_ce']:+.6f}, "
                f"fraction beneficial-to-keep={100*r['fraction_delta_gt_0']:.1f}%, "
                f"fraction |ΔCE|<.001={100*r['fraction_abs_delta_lt_0.001']:.1f}%, "
                f"rho(RMS, ΔCE)={r['rms_vs_delta_ce_spearman']:.3f}.\n"
            )
        lines.append("Positive ΔCE means removing the head hurt MLM CE. This is more direct evidence of usefulness than RMS alone.\n")

    lines.append("\n## 4. Post-hoc head-count curve\n")
    if curve is None:
        lines.append("Skipped. Re-run without `--skip-head-curve`.\n")
    else:
        lines.append("This removes heads from a model trained dense. It tests redundancy/robustness, **not** whether a separately trained smaller model would match it.\n")
        for k in sorted(int(x) for x in curve["aggregate"].keys()):
            a = curve["aggregate"][str(k)] if str(k) in curve["aggregate"] else curve["aggregate"][k]
            parts = []
            for method, stats in a.items():
                parts.append(f"{method}={stats['mean']:.6f}" + (f"±{stats['std']:.6f}" if stats['std'] > 0 else ""))
            lines.append(f"- keep {k}/{H}: " + ", ".join(parts) + "\n")

    lines.append("\n## 5. Assumptions this audit directly tests\n")
    lines.append("1. **'Dense heads should have good effective rank.'** Check raw AND directional rank before treating rank as an optimization target.\n")
    lines.append("2. **'Low effective rank means duplicate heads.'** False if directional rank/cosine are healthy; it may just be energy imbalance.\n")
    lines.append("3. **'The attention computation is where collapse happens.'** Context-vs-residual rank tells us whether W_O is actually the bottleneck.\n")
    lines.append("4. **'A large residual head is an important head.'** Exact ΔCE vs RMS tests that directly.\n")
    lines.append("5. **'All 16 heads are needed because the model was trained dense.'** The ablations/head-count curve test this rather than assuming it.\n")
    lines.append("6. **'Keeping the biggest heads is enough.'** Top-energy vs random same-count subsets tests whether magnitude contains useful pruning information.\n")
    lines.append("7. **'Effective rank should be directly maximized.'** Only justified if low directional rank associates with worse CE; raw rank alone also penalizes legitimate specialization.\n")

    with (output_dir / "summary.md").open("w") as f:
        f.writelines(lines)


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(description="Audit assumptions in the vanilla dense-16 HELM baseline")
    parser.add_argument("--model-repo", default=MODEL_REPO)
    parser.add_argument("--checkpoint", default=CHECKPOINT_FILE,
                        help="checkpoint filename or 'latest' (default)")
    parser.add_argument("--data-repo", default=DATA_REPO)
    parser.add_argument("--validation-file", default=VALIDATION_FILE)
    parser.add_argument("--device", default="auto", choices=["auto", "xla", "cuda", "cpu"])
    parser.add_argument("--num-examples", type=int, default=16)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--seq-len", type=int, default=1024)
    parser.add_argument("--seed", type=int, default=1216)
    parser.add_argument("--functional-batches", type=int, default=4)
    parser.add_argument("--functional-sample-tokens", type=int, default=16)
    parser.add_argument("--exact-ablation", action="store_true")
    parser.add_argument("--ablation-layers", default="0,5,11")
    parser.add_argument("--ablation-batches", type=int, default=1)
    parser.add_argument("--skip-head-curve", action="store_true")
    parser.add_argument("--head-curve-batches", type=int, default=2)
    parser.add_argument("--head-curve-trials", type=int, default=2)
    parser.add_argument("--keep-counts", default="4,8,12,16")
    parser.add_argument("--output-dir", default="dense16_assumption_audit")
    parser.add_argument("--cache-dir", default="./dense16_audit_cache")
    args = parser.parse_args()

    seed_everything(args.seed)
    dev = resolve_device(args.device)
    token = get_hf_token()
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path, validation_path, checkpoint_name = download_assets(
        Path(args.cache_dir), token, args.model_repo, args.checkpoint,
        args.data_repo, args.validation_file,
    )
    checkpoint_step = checkpoint_step_from_name(checkpoint_name)
    model, config = load_model(ckpt_path, dev)

    if config.num_attention_heads != 16:
        print(f"WARNING: model.py says {config.num_attention_heads} heads, not 16. "
              "The script will continue generically, but this is not the expected vanilla baseline.")

    batches = prepare_batches(
        validation_path, config, args.num_examples, args.batch_size,
        args.seq_len, args.seed,
    )

    print("\n[1/4] Baseline CE...")
    baseline_ce = evaluate_ce(model, batches, dev, checkpoint_step)
    print(f"Baseline dense CE = {baseline_ce:.6f}")

    print("\n[2/4] All-layer head geometry / effective rank...")
    head_metrics = head_bank_audit(
        model, batches, dev, checkpoint_step,
        min(args.functional_batches, len(batches)),
        args.functional_sample_tokens, output_dir,
    )
    for li in sorted(head_metrics):
        r = head_metrics[li]
        print(
            f"L{li:02d}: residual raw rank={r['residual_raw_entropy_rank']:.2f}/16, "
            f"directional rank={r['residual_directional_entropy_rank']:.2f}/16, "
            f"mean|cos|={r['residual_cosine']['mean_abs']:.3f}, "
            f"top1 energy={100*r['residual_top1_energy_fraction']:.1f}%"
        )

    exact = None
    if args.exact_ablation:
        print("\n[3/4] Exact single-head ablation...")
        layers = parse_layers(args.ablation_layers, len(model.model.blocks))
        exact = exact_single_head_ablation(
            model, batches, dev, checkpoint_step, layers,
            min(args.ablation_batches, len(batches)), head_metrics, output_dir,
        )
    else:
        print("\n[3/4] Exact ablation skipped (use --exact-ablation).")

    curve = None
    if not args.skip_head_curve:
        print("\n[4/4] Post-hoc head-count curve...")
        keep_counts = [int(x.strip()) for x in args.keep_counts.split(",") if x.strip()]
        curve = evaluate_posthoc_head_curve(
            model, batches, dev, checkpoint_step, head_metrics,
            keep_counts, args.head_curve_trials,
            min(args.head_curve_batches, len(batches)), args.seed, output_dir,
        )
    else:
        print("\n[4/4] Head-count curve skipped.")

    payload = {
        "model_repo": args.model_repo,
        "checkpoint": checkpoint_name,
        "checkpoint_step": checkpoint_step,
        "architecture": {
            "hidden_size": config.hidden_size,
            "num_attention_heads": config.num_attention_heads,
            "d_head": getattr(config, "d_head", None) or (config.hidden_size // config.num_attention_heads),
            "total_head_dim": config.num_attention_heads * (getattr(config, "d_head", None) or (config.hidden_size // config.num_attention_heads)),
        },
        "baseline_ce": baseline_ce,
        "head_metrics": head_metrics,
        "exact_ablation": exact,
        "posthoc_head_curve": curve,
    }
    with (output_dir / "summary.json").open("w") as f:
        json.dump(json_safe(payload), f, indent=2)

    write_summary(output_dir, args, checkpoint_name, config, baseline_ce, head_metrics, exact, curve)

    archive_base = output_dir.parent / f"{output_dir.name}_results"
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=output_dir)

    print("\n" + "=" * 80)
    print("DENSE-16 ASSUMPTION AUDIT COMPLETE")
    print(f"Summary: {output_dir / 'summary.md'}")
    print(f"JSON:    {output_dir / 'summary.json'}")
    print(f"ZIP:     {archive_path}")


if __name__ == "__main__":
    main()


Overwriting audit_dense16.py


## 3. Write the rank-over-training audit

This tests another assumption: that an early checkpoint's head geometry represents the architecture. It automatically finds available checkpoints in the Hugging Face repository and samples them across training.


In [4]:
%%writefile audit_dense16_rank_over_time.py
#!/usr/bin/env python3
"""
Track dense-16 head geometry over training checkpoints.

This specifically tests an assumption we have repeatedly made: that the head-rank
profile at one early checkpoint (e.g. 6.5k) characterizes the architecture.

For a handful of checkpoints from the SAME dense-16 run, on the SAME validation
examples/masking, record:
  - MLM CE
  - raw residual effective rank
  - directional residual effective rank
  - pre-W_O context raw/directional rank
  - top-1/top-4 residual energy share
  - mean absolute head cosine
  - energy Gini

No ablation curves or router tests are run here; it is intentionally lightweight.
"""

from __future__ import annotations

import argparse
import csv
import json
import re
import shutil
from pathlib import Path
from typing import List, Optional

import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, hf_hub_download

from audit_dense16 import (
    MODEL_REPO, DATA_REPO, VALIDATION_FILE,
    HELMConfig,
    get_hf_token, seed_everything, resolve_device,
    prepare_batches, load_model, evaluate_ce, head_bank_audit,
    checkpoint_step_from_name, json_safe,
)


def list_checkpoints(repo_id: str, token: Optional[str]):
    api = HfApi(token=token)
    files = api.list_repo_files(repo_id=repo_id, repo_type="model", token=token)
    out = []
    for name in files:
        m = re.search(r"checkpoint-(\d+)\.pt$", name)
        if m:
            out.append((int(m.group(1)), name))
    return sorted(out)


def choose_evenly(checkpoints, n: int):
    if len(checkpoints) <= n:
        return checkpoints
    idx = np.linspace(0, len(checkpoints) - 1, num=n).round().astype(int)
    idx = sorted(set(idx.tolist()))
    return [checkpoints[i] for i in idx]


def parse_requested(spec: Optional[str], available):
    if not spec:
        return None
    by_step = {step: (step, name) for step, name in available}
    chosen = []
    for part in spec.split(","):
        part = part.strip()
        if not part:
            continue
        if part.startswith("checkpoint-"):
            m = re.search(r"checkpoint-(\d+)\.pt$", part)
            if not m:
                raise ValueError(f"Bad checkpoint name: {part}")
            step = int(m.group(1))
        else:
            step = int(part)
        if step not in by_step:
            raise ValueError(f"Checkpoint step {step} not found in repository")
        chosen.append(by_step[step])
    return chosen


def plot_metric(rows, key, ylabel, path, layers):
    fig, ax = plt.subplots(figsize=(8, 4))
    for li in layers:
        rr = [r for r in rows if r["layer"] == li]
        rr.sort(key=lambda r: r["step"])
        ax.plot([r["step"] for r in rr], [r[key] for r in rr], marker="o", label=f"L{li}")
    ax.set_xlabel("Training step")
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + " over training")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def main():
    p = argparse.ArgumentParser(description="Track dense-16 effective rank over training")
    p.add_argument("--model-repo", default=MODEL_REPO)
    p.add_argument("--data-repo", default=DATA_REPO)
    p.add_argument("--validation-file", default=VALIDATION_FILE)
    p.add_argument("--device", default="auto", choices=["auto", "xla", "cuda", "cpu"])
    p.add_argument("--num-checkpoints", type=int, default=5,
                   help="Evenly sample this many available checkpoints")
    p.add_argument("--checkpoints", default=None,
                   help="Optional comma-separated steps, e.g. 6500,13000,32500,65000")
    p.add_argument("--layers", default="0,5,11")
    p.add_argument("--num-examples", type=int, default=8)
    p.add_argument("--batch-size", type=int, default=2)
    p.add_argument("--seq-len", type=int, default=1024)
    p.add_argument("--functional-batches", type=int, default=2)
    p.add_argument("--functional-sample-tokens", type=int, default=16)
    p.add_argument("--seed", type=int, default=1216)
    p.add_argument("--cache-dir", default="./dense16_rank_time_cache")
    p.add_argument("--output-dir", default="dense16_rank_over_time")
    args = p.parse_args()

    seed_everything(args.seed)
    token = get_hf_token()
    dev = resolve_device(args.device)
    cache = Path(args.cache_dir)
    output = Path(args.output_dir)
    cache.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)

    available = list_checkpoints(args.model_repo, token)
    if not available:
        raise RuntimeError(f"No checkpoint-XXXXXX.pt files found in {args.model_repo}")

    requested = parse_requested(args.checkpoints, available)
    selected = requested if requested is not None else choose_evenly(available, args.num_checkpoints)
    print("Selected checkpoints:")
    for step, name in selected:
        print(f"  {step:>7}: {name}")

    validation_path = hf_hub_download(
        repo_id=args.data_repo,
        filename=args.validation_file,
        repo_type="dataset",
        token=token,
        local_dir=str(cache / "dataset"),
    )

    # Build the exact same deterministic examples ONCE for every checkpoint.
    config_for_data = HELMConfig()
    batches = prepare_batches(
        Path(validation_path), config_for_data, args.num_examples,
        args.batch_size, args.seq_len, args.seed,
    )

    layers = [int(x.strip()) for x in args.layers.split(",") if x.strip()]
    rows = []
    ce_rows = []

    for step, name in selected:
        print("\n" + "=" * 80)
        print(f"CHECKPOINT {name}")
        ckpt_path = hf_hub_download(
            repo_id=args.model_repo,
            filename=name,
            repo_type="model",
            token=token,
            local_dir=str(cache / "model_repo"),
        )

        model, config = load_model(Path(ckpt_path), dev)
        ce = evaluate_ce(model, batches, dev, checkpoint_step_from_name(name))
        ce_rows.append({"step": step, "checkpoint": name, "ce": ce})
        print(f"CE={ce:.6f}")

        ckpt_out = output / f"step_{step:06d}"
        ckpt_out.mkdir(parents=True, exist_ok=True)
        metrics = head_bank_audit(
            model, batches, dev, step,
            min(args.functional_batches, len(batches)),
            args.functional_sample_tokens, ckpt_out, layers=layers,
        )

        for li in layers:
            r = metrics[li]
            row = {
                "step": step,
                "checkpoint": name,
                "layer": li,
                "ce": ce,
                "context_raw_rank": r["context_raw_entropy_rank"],
                "context_directional_rank": r["context_directional_entropy_rank"],
                "residual_raw_rank": r["residual_raw_entropy_rank"],
                "residual_directional_rank": r["residual_directional_entropy_rank"],
                "mean_abs_residual_cosine": r["residual_cosine"]["mean_abs"],
                "top1_energy_fraction": r["residual_top1_energy_fraction"],
                "top4_energy_fraction": r["residual_top4_energy_fraction"],
                "energy_gini": r["residual_energy_gini"],
                "components_90pct": r["residual_components_90pct"],
            }
            rows.append(row)
            print(
                f"L{li}: raw={row['residual_raw_rank']:.2f}/16, "
                f"dir={row['residual_directional_rank']:.2f}/16, "
                f"top1={100*row['top1_energy_fraction']:.1f}%"
            )

        del model

    with (output / "rank_over_time.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader(); writer.writerows(rows)
    with (output / "ce_over_time.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=ce_rows[0].keys())
        writer.writeheader(); writer.writerows(ce_rows)

    plot_metric(rows, "residual_raw_rank", "Residual raw effective rank", output / "raw_rank_over_time.png", layers)
    plot_metric(rows, "residual_directional_rank", "Residual directional effective rank", output / "directional_rank_over_time.png", layers)
    plot_metric(rows, "top1_energy_fraction", "Top-1 residual energy fraction", output / "top1_energy_over_time.png", layers)
    plot_metric(rows, "mean_abs_residual_cosine", "Mean |residual head cosine|", output / "cosine_over_time.png", layers)

    # Compact markdown summary.
    lines = [
        "# Dense-16 Rank Over Training\n",
        f"Repository: `{args.model_repo}`\n",
        "Same validation examples and deterministic masks are used for every checkpoint.\n",
        "\n|Step|CE|Layer|Raw rank|Directional rank|Mean |cos||Top-1 energy|Top-4 energy|\n",
        "|---:|---:|---:|---:|---:|---:|---:|---:|\n",
    ]
    for r in rows:
        lines.append(
            f"|{r['step']}|{r['ce']:.6f}|{r['layer']}|{r['residual_raw_rank']:.2f}/16|"
            f"{r['residual_directional_rank']:.2f}/16|{r['mean_abs_residual_cosine']:.3f}|"
            f"{100*r['top1_energy_fraction']:.1f}%|{100*r['top4_energy_fraction']:.1f}%|\n"
        )
    lines += [
        "\n## Why this matters\n",
        "- If directional rank rises substantially with training, an early low-rank checkpoint should not be treated as architectural collapse.\n",
        "- If raw rank falls while directional rank stays high, specialization is becoming more unequal in magnitude rather than more redundant in direction.\n",
        "- If both ranks fall as CE improves, maximizing rank may conflict with task optimization rather than help it.\n",
        "- If both ranks fall while CE worsens/plateaus, rank/diversity becomes a much stronger intervention target.\n",
    ]
    (output / "summary.md").write_text("".join(lines))

    payload = {"ce": ce_rows, "rank": rows, "selected_checkpoints": selected}
    (output / "summary.json").write_text(json.dumps(json_safe(payload), indent=2))

    archive_base = output.parent / f"{output.name}_results"
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=output)
    print("\nDONE")
    print(f"Summary: {output / 'summary.md'}")
    print(f"ZIP: {archive_path}")


if __name__ == "__main__":
    main()


Writing audit_dense16_rank_over_time.py


## 4. Single-checkpoint audit

For an apples-to-apples comparison with HELM_7c, start with **checkpoint 6,500**. Exact ablation is restricted to layers 0, 5, and 11 to keep the intervention interpretable and reasonably sized.

The post-hoc head-count curve keeps 4/8/12/16 heads per layer using either the largest measured residual-energy heads or random same-count subsets. Remember: this is a pruning/robustness test, **not** a substitute for training a smaller model from scratch.


In [9]:
import subprocess, sys

CHECKPOINT = "checkpoint-006500.pt"   # use "latest" if you want the newest checkpoint instead
DEVICE = "xla"

cmd = [
    sys.executable, "audit_dense16.py",
    "--device", DEVICE,
    "--checkpoint", CHECKPOINT,
    "--num-examples", "16",
    "--batch-size", "2",
    "--functional-batches", "4",
    "--functional-sample-tokens", "16",
    "--exact-ablation",
    "--ablation-layers", "0,5,11",
    "--ablation-batches", "1",
    "--head-curve-batches", "2",
    "--head-curve-trials", "2",
    "--keep-counts", "4,8,12,16",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)


/usr/local/bin/python audit_dense16.py --device xla --checkpoint checkpoint-006500.pt --num-examples 16 --batch-size 2 --functional-batches 4 --functional-sample-tokens 16 --exact-ablation --ablation-layers 0,5,11 --ablation-batches 1 --head-curve-batches 2 --head-curve-trials 2 --keep-counts 4,8,12,16


/kaggle/working/audit_dense16.py:292: DeprecationWarning: Use torch_xla.device instead
  return DeviceContext(xm.xla_device(), "xla", xm=xm, autocast_dtype=torch.bfloat16)
E0000 00:00:1786486242.713365    3276 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238
/kaggle/working/audit_dense16.py:268: DeprecationWarning: Use torch_xla.sync instead
  self.xm.mark_step()
/kaggle/working/audit_dense16.py:268: DeprecationWarning: Use torch_xla.sync instead
  self.xm.mark_step()


Architecture from model.py: H=16, hidden=1024, d_head=None
Prepared 16 examples -> 8 batches, seq_len=1024

[1/4] Baseline CE...
Baseline dense CE = 3.494338

[2/4] All-layer head geometry / effective rank...
L00: residual raw rank=7.17/16, directional rank=9.74/16, mean|cos|=0.347, top1 energy=19.9%
L01: residual raw rank=7.93/16, directional rank=12.35/16, mean|cos|=0.223, top1 energy=39.4%
L02: residual raw rank=9.10/16, directional rank=14.78/16, mean|cos|=0.102, top1 energy=32.6%
L03: residual raw rank=12.94/16, directional rank=15.61/16, mean|cos|=0.042, top1 energy=20.7%
L04: residual raw rank=11.61/16, directional rank=15.88/16, mean|cos|=0.022, top1 energy=19.7%
L05: residual raw rank=11.61/16, directional rank=15.86/16, mean|cos|=0.025, top1 energy=19.2%
L06: residual raw rank=10.56/16, directional rank=15.49/16, mean|cos|=0.031, top1 energy=24.3%
L07: residual raw rank=10.67/16, directional rank=15.84/16, mean|cos|=0.026, top1 energy=24.1%
L08: residual raw rank=12.46/16, di

CompletedProcess(args=['/usr/local/bin/python', 'audit_dense16.py', '--device', 'xla', '--checkpoint', 'checkpoint-006500.pt', '--num-examples', '16', '--batch-size', '2', '--functional-batches', '4', '--functional-sample-tokens', '16', '--exact-ablation', '--ablation-layers', '0,5,11', '--ablation-batches', '1', '--head-curve-batches', '2', '--head-curve-trials', '2', '--keep-counts', '4,8,12,16'], returncode=0)

## 5. Rank over training

This uses fewer examples/batches because it repeats the same geometry analysis across multiple checkpoints. By default it chooses five available checkpoints evenly across the run and measures layers 0, 5, and 11.


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "audit_dense16_rank_over_time.py",
    "--device", "xla",
    "--num-checkpoints", "5",
    "--layers", "0,5,11",
    "--num-examples", "8",
    "--batch-size", "2",
    "--functional-batches", "2",
    "--functional-sample-tokens", "16",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)


## Outputs to send back

For the single-checkpoint audit, send:

- `dense16_assumption_audit/summary.md`
- `dense16_assumption_audit_results.zip`

For the training-dynamics audit, send:

- `dense16_rank_over_time/summary.md`
- `dense16_rank_over_time_results.zip`

The most important comparisons will be **raw vs directional rank**, **context vs residual rank**, **RMS vs exact ΔCE**, and **how all of those quantities move as CE improves during training**.
